# 11 Conversational Memory

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `11-conversational-memory.ipynb`

In [2]:
# Start coding here
# ==================================================
# Notebook 11
# Conversational Memory
# ==================================================

import json
import pandas as pd

from transformers import pipeline

In [3]:
query_rewriter_llm = pipeline("text2text-generation", model="google/flan-t5-base")

Device set to use cpu


In [4]:
conversation_memory = []

In [5]:
def add_message(role, content):

    conversation_memory.append({"role": role, "content": content})

In [6]:
def build_history_context(max_turns=6):

    history = conversation_memory[-max_turns:]

    text = ""

    for item in history:

        text += f"{item['role']}: " f"{item['content']}\n"

    return text

In [7]:
add_message("user", "What was revenue in 2026?")

add_message("assistant", "Revenue was $5.2 billion.")

print(build_history_context())

user: What was revenue in 2026?
assistant: Revenue was $5.2 billion.



In [8]:
def rewrite_query(current_question, conversation_context):

    prompt = f"""
Given the conversation history and the latest user question,
rewrite the latest question into a standalone question.

Conversation:

{conversation_context}

Question:

{current_question}

Standalone Question:
"""

    response = query_rewriter_llm(prompt, max_new_tokens=64)

    return response[0]["generated_text"]

In [9]:
history = """
user: What was revenue in 2026?
assistant: Revenue was $5.2 billion.
"""

question = "Did it increase?"

In [10]:
rewrite_query(question, history)

'What was the revenue in 2026?'

In [11]:
def memory_aware_query(question):

    history = build_history_context()

    standalone_question = rewrite_query(question, history)

    return standalone_question

In [12]:
memory_aware_query("Did it increase?")

'What was the revenue in 2026?'

In [13]:
def summarize_memory():

    history = build_history_context()

    prompt = f"""
Summarize this conversation.

{history}
"""

    response = query_rewriter_llm(prompt, max_new_tokens=128)

    return response[0]["generated_text"]

In [14]:
summarize_memory()

'user: Revenue in 2026 was $5.2 billion.'

In [15]:
def conversational_retrieve(user_question):

    standalone_question = memory_aware_query(user_question)

    return standalone_question

In [16]:
conversational_retrieve("Did it increase?")

'What was the revenue in 2026?'

In [17]:
conversation_state = {
    "session_id": "abc123",
    "messages": [],
    "summary": "",
    "active_documents": [],
}

In [18]:
with open("artifacts/conversation_memory.json", "w") as file:

    json.dump(conversation_memory, file, indent=4)

In [19]:
with open("artifacts/conversation_memory.json", "r") as file:

    loaded_memory = json.load(file)

loaded_memory[:2]

[{'role': 'user', 'content': 'What was revenue in 2026?'},
 {'role': 'assistant', 'content': 'Revenue was $5.2 billion.'}]

In [20]:
from transformers import pipeline

query_rewriter_llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vinna\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [21]:
model = "microsoft/Phi-3-mini-4k-instruct"